# Phase 1 — Do skeletons leak identity?

**NTU RGB+D 60 · ST-GCN · action recognition vs. re-identification**

This notebook produces the foundational result of the paper: a measurement of how much
personal identity survives in "privacy-preserving" skeleton data.

**Three experiments**

| | Question | Expected |
|---|---|---|
| A | How well can we recognise the *action*? (utility) | ~80–85% |
| B | How well can we recognise the *person*? (leakage) | ~70–85% |
| C | Does the action model's own feature space leak identity? | high = the key finding |

**Before running:** `ntu60_3danno.pkl` (1.1 GB) must be at `PhD_Q1/data/` in your Drive.

**Runtime:** ~2–4 h on a T4/P100. All training checkpoints every epoch, so you can
stop and re-run any cell — it resumes automatically.

## 0 · Setup

In [ ]:
import os, sys, re, json, time, math, pickle, random
import numpy as np

SEED = 1
random.seed(SEED); np.random.seed(SEED)

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/PhD_Q1'
    SCRATCH = '/content/scratch'
elif IN_KAGGLE:
    PROJECT = '/kaggle/working/PhD_Q1'
    SCRATCH = '/kaggle/working/scratch'
else:
    PROJECT = os.path.abspath('./PhD_Q1')
    SCRATCH = os.path.abspath('./scratch')

DATA_DIR = os.path.join(PROJECT, 'data')
CKPT_DIR = os.path.join(PROJECT, 'checkpoints')
RES_DIR  = os.path.join(PROJECT, 'results')
for d in (DATA_DIR, CKPT_DIR, RES_DIR, SCRATCH):
    os.makedirs(d, exist_ok=True)

PKL_PATH  = os.path.join(DATA_DIR, 'ntu60_3danno.pkl')
CACHE_X   = os.path.join(DATA_DIR, 'ntu60_T64_x.npy')
CACHE_M   = os.path.join(DATA_DIR, 'ntu60_T64_meta.npz')

# geometry / tensor shapes
NUM_NODE, MAX_PERSON, T_LEN, IN_CH = 25, 2, 64, 3

print('project :', PROJECT)
print('pkl     :', PKL_PATH, '->', os.path.exists(PKL_PATH))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. Colab: Runtime > Change runtime type > GPU.')

## 1 · Load and verify

Confirms the file is intact and that **identity labels are recoverable**. The sample name
`S001C001P001R001A001` encodes setup, Camera, **Person**, Repetition and Action.

In [ ]:
t0 = time.time()
with open(PKL_PATH, 'rb') as f:
    D = pickle.load(f)
print('loaded in %.1f s' % (time.time() - t0))

anns  = D['annotations']
names = [a['frame_dir'] for a in anns]

setup   = np.array([int(n[1:4])   for n in names], dtype=np.int16)
camera  = np.array([int(n[5:8])   for n in names], dtype=np.int16)
subject = np.array([int(n[9:12])  for n in names], dtype=np.int16)
action  = np.array([a['label']    for a in anns],  dtype=np.int16)

print('samples  :', len(anns))
print('subjects :', len(np.unique(subject)))
print('cameras  :', np.unique(camera))
print('actions  :', len(np.unique(action)))
print('example  :', names[0])
print('keys     :', list(anns[0].keys()))
print('keypoint :', np.asarray(anns[0]['keypoint']).shape, ' (M, T, V, C)')

assert len(np.unique(subject)) == 40, 'expected 40 subjects'
print('\nOK — identity labels present. Proceed.')

## 2 · Preprocessing

Each sequence is normalised the standard way: blank frames removed, body centred on the
spine, skeleton rotated to a canonical orientation, then resampled to 64 frames and padded
to 2 people. Cached once to `.npy` (~1.1 GB) so later cells load in seconds.

In [ ]:
def _unit(v):
    n = np.linalg.norm(v)
    return v if n < 1e-8 else v / n

def _rotation_matrix(axis, theta):
    axis = _unit(np.asarray(axis, dtype=np.float64))
    a = math.cos(theta / 2.0)
    b, c, d = -axis * math.sin(theta / 2.0)
    aa, bb, cc, dd = a*a, b*b, c*c, d*d
    bc, ad, ac, ab, bd, cd = b*c, a*d, a*c, a*b, b*d, c*d
    return np.array([[aa+bb-cc-dd, 2*(bc+ad),   2*(bd-ac)],
                     [2*(bc-ad),   aa+cc-bb-dd, 2*(cd+ab)],
                     [2*(bd+ac),   2*(cd-ab),   aa+dd-bb-cc]], dtype=np.float32)

def _angle_between(v1, v2):
    return np.arccos(np.clip(np.dot(_unit(v1), _unit(v2)), -1.0, 1.0))

def _align(skel, ja, jb, target):
    # rotate the whole sequence so that joint ja -> jb points along `target`
    vec = skel[0, 0, jb] - skel[0, 0, ja]
    if np.linalg.norm(vec) < 1e-8:
        return skel
    axis, angle = np.cross(vec, target), _angle_between(vec, target)
    if np.linalg.norm(axis) < 1e-8 or angle < 1e-8:
        return skel
    return (skel @ _rotation_matrix(axis, angle).T).astype(np.float32)

def pre_normalize(kp):
    kp = np.asarray(kp, dtype=np.float32).copy()
    valid = kp.sum(axis=(0, 2, 3)) != 0
    if valid.any():
        kp = kp[:, valid]
    if kp.shape[1] == 0:
        return np.zeros((1, 1, NUM_NODE, 3), dtype=np.float32)
    mask = kp.sum(axis=-1) != 0
    kp[mask] = kp[mask] - kp[0, 0, 1].copy()          # centre on spine
    kp = _align(kp, 0, 1, np.array([0, 0, 1], np.float32))   # spine     -> z
    kp = _align(kp, 8, 4, np.array([1, 0, 0], np.float32))   # shoulders -> x
    return kp

def resample(kp, t_len=T_LEN):
    M, T, V, Ch = kp.shape
    out = kp[:, np.linspace(0, T - 1, t_len).round().astype(np.int64)]
    if M < MAX_PERSON:
        out = np.concatenate(
            [out, np.zeros((MAX_PERSON - M, t_len, V, Ch), out.dtype)], axis=0)
    return out[:MAX_PERSON]

# quick self-test on synthetic data
_t = np.random.RandomState(0).normal(size=(1, 90, 25, 3)).astype(np.float32)
assert resample(pre_normalize(_t)).shape == (MAX_PERSON, T_LEN, NUM_NODE, 3)
print('preprocessing OK')

In [ ]:
if os.path.exists(CACHE_X) and os.path.exists(CACHE_M):
    print('cache found, skipping preprocessing')
else:
    N = len(anns)
    X = np.zeros((N, MAX_PERSON, T_LEN, NUM_NODE, IN_CH), dtype=np.float16)
    t0 = time.time()
    for i, a in enumerate(anns):
        X[i] = resample(pre_normalize(np.asarray(a['keypoint']))).astype(np.float16)
        if (i + 1) % 5000 == 0:
            el = time.time() - t0
            print('  %6d / %d   %.0f s elapsed, ~%.0f s left'
                  % (i + 1, N, el, el / (i + 1) * (N - i - 1)))
    np.save(CACHE_X, X)
    np.savez(CACHE_M, setup=setup, camera=camera, subject=subject,
             action=action, names=np.array(names))
    print('cached in %.0f s -> %.2f GB' % (time.time() - t0, X.nbytes / 1e9))
    del X

In [ ]:
# copy to local disk first: Drive I/O is slow for repeated reads
LOCAL_X = os.path.join(SCRATCH, 'x.npy')
if not os.path.exists(LOCAL_X):
    import shutil; shutil.copy(CACHE_X, LOCAL_X)

X = np.load(LOCAL_X)                       # (N, 2, 64, 25, 3) float16
_m = np.load(CACHE_M, allow_pickle=True)
setup, camera, subject, action = _m['setup'], _m['camera'], _m['subject'], _m['action']
names = list(_m['names'])

SUBJ_IDS   = sorted(np.unique(subject).tolist())
SUBJ_LABEL = np.array([SUBJ_IDS.index(s) for s in subject], dtype=np.int64)  # 0..39
ACT_LABEL  = action.astype(np.int64)                                         # 0..59

print('X', X.shape, X.dtype, '%.2f GB' % (X.nbytes / 1e9))
print('action classes', ACT_LABEL.max() + 1, '| identity classes', SUBJ_LABEL.max() + 1)

## 3 · Evaluation protocols

Three splits, and the distinction between them matters for the paper.

| Split | Train | Test | Used for |
|---|---|---|---|
| `xsub` | 20 subjects | the other 20 | **Action** (standard NTU benchmark) |
| `reid_rand` | 80% of each person's clips | remaining 20% | **Identity**, standard protocol |
| `reid_xview` | cameras 1 and 2 | camera 3 | **Identity**, harder — unseen viewpoint |

Identity cannot be evaluated on `xsub`: its train and test subjects are disjoint by
construction, so the label does not exist at test time. Reporting `reid_xview` alongside
`reid_rand` pre-empts the reviewer objection that leakage is merely near-duplicate clips.

In [ ]:
XSUB_TRAIN = [1,2,4,5,8,9,13,14,15,16,17,18,19,25,27,28,31,34,35,38]

SPLITS = {}
SPLITS['xsub'] = (np.where(np.isin(subject, XSUB_TRAIN))[0],
                  np.where(~np.isin(subject, XSUB_TRAIN))[0])

rs, tr, te = np.random.RandomState(SEED), [], []
for s in SUBJ_IDS:                                   # stratified 80/20 per person
    idx = np.where(subject == s)[0]; rs.shuffle(idx)
    k = int(round(0.8 * len(idx)))
    tr += idx[:k].tolist(); te += idx[k:].tolist()
SPLITS['reid_rand'] = (np.array(sorted(tr)), np.array(sorted(te)))

SPLITS['reid_xview'] = (np.where(np.isin(camera, [1, 2]))[0],
                        np.where(camera == 3)[0])

for k, (a, b) in SPLITS.items():
    print('%-11s train %6d  test %6d  | subjects %2d / %2d'
          % (k, len(a), len(b), len(set(subject[a])), len(set(subject[b]))))

assert len(set(subject[SPLITS['xsub'][0]]) & set(subject[SPLITS['xsub'][1]])) == 0
assert len(set(subject[SPLITS['reid_xview'][0]]) &
           set(subject[SPLITS['reid_xview'][1]])) == 40

## 4 · ST-GCN

Ten spatial-temporal graph convolution blocks over the NTU 25-joint skeleton, with the
standard three-subset partition (self / inward / outward). `extract()` returns the 256-d
representation — that is what Experiment C attacks.

In [ ]:
_INWARD_1B = [(1,2),(2,21),(3,21),(4,3),(5,21),(6,5),(7,6),(8,7),(9,21),
              (10,9),(11,10),(12,11),(13,1),(14,13),(15,14),(16,15),
              (17,1),(18,17),(19,18),(20,19),(22,23),(23,8),(24,25),(25,12)]
INWARD    = [(i-1, j-1) for i, j in _INWARD_1B]
OUTWARD   = [(j, i) for i, j in INWARD]
SELF_LINK = [(i, i) for i in range(NUM_NODE)]

def _edge2mat(link, n):
    A = np.zeros((n, n), dtype=np.float32)
    for i, j in link:
        A[j, i] = 1.0
    return A

def _norm_digraph(A):
    Dl, Dn = A.sum(0), np.zeros_like(A)
    for i in range(A.shape[0]):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i] ** -1.0
    return A @ Dn

def build_adjacency():
    return np.stack([_edge2mat(SELF_LINK, NUM_NODE),
                     _norm_digraph(_edge2mat(INWARD, NUM_NODE)),
                     _norm_digraph(_edge2mat(OUTWARD, NUM_NODE))]).astype(np.float32)


class SpatialGC(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.K = A.shape[0]
        self.conv = nn.Conv2d(in_c, out_c * self.K, 1)
        self.register_buffer('A', torch.from_numpy(A.copy()))

    def forward(self, x):
        N, _, T, V = x.shape
        x = self.conv(x).view(N, self.K, -1, T, V)
        return torch.einsum('nkctv,kvw->nctw', x, self.A.to(x.dtype)).contiguous()


class STGCNBlock(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, residual=True):
        super().__init__()
        self.gcn = SpatialGC(in_c, out_c, A)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.tcn = nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0))
        self.bn2 = nn.BatchNorm2d(out_c)
        self.drop = nn.Dropout(0.1)
        if not residual:
            self.res = None
        elif in_c == out_c and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)),
                                     nn.BatchNorm2d(out_c))

    def forward(self, x):
        r = 0 if self.res is None else self.res(x)
        y = F.relu(self.bn1(self.gcn(x)))
        y = self.drop(self.bn2(self.tcn(y)))
        return F.relu(y + r)


class STGCN(nn.Module):
    def __init__(self, num_class, in_channels=IN_CH, num_person=MAX_PERSON, dropout=0.3):
        super().__init__()
        A = build_adjacency()
        self.data_bn = nn.BatchNorm1d(num_person * in_channels * NUM_NODE)
        cfg = [(in_channels, 64, 1, False), (64, 64, 1, True), (64, 64, 1, True),
               (64, 64, 1, True), (64, 128, 2, True), (128, 128, 1, True),
               (128, 128, 1, True), (128, 256, 2, True), (256, 256, 1, True),
               (256, 256, 1, True)]
        self.layers = nn.ModuleList([STGCNBlock(i, o, A, s, r) for i, o, s, r in cfg])
        self.drop, self.fc, self.feat_dim = nn.Dropout(dropout), nn.Linear(256, num_class), 256

    def extract(self, x):                      # x: N, C, T, V, M
        N, C, T, V, M = x.shape
        x = x.permute(0, 4, 3, 1, 2).contiguous().view(N, M * V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T).permute(0, 1, 3, 4, 2).contiguous().view(N * M, C, T, V)
        for l in self.layers:
            x = l(x)
        return F.adaptive_avg_pool2d(x, 1).view(N, M, -1).mean(1)

    def forward(self, x, return_feat=False):
        f = self.extract(x)
        out = self.fc(self.drop(f))
        return (out, f) if return_feat else out


_m = STGCN(60)
_o, _f = _m(torch.randn(2, IN_CH, T_LEN, NUM_NODE, MAX_PERSON), return_feat=True)
print('logits', tuple(_o.shape), '| features', tuple(_f.shape))
print('parameters: %.2f M' % (sum(p.numel() for p in _m.parameters()) / 1e6))
del _m, _o, _f

## 5 · Training engine (checkpoints every epoch, resumes automatically)

In [ ]:
def _euler(theta):
    cx, cy, cz = np.cos(theta); sx, sy, sz = np.sin(theta)
    Rx = np.array([[1,0,0],[0,cx,-sx],[0,sx,cx]])
    Ry = np.array([[cy,0,sy],[0,1,0],[-sy,0,cy]])
    Rz = np.array([[cz,-sz,0],[sz,cz,0],[0,0,1]])
    return (Rz @ Ry @ Rx).astype(np.float32)

def augment(x, rot=0.3, scale=0.1):
    R = _euler(np.random.uniform(-rot, rot, 3))
    s = 1.0 + np.random.uniform(-scale, scale)
    mask = (x.sum(-1) != 0)[..., None]
    return np.where(mask, (x @ R.T) * s, 0.0).astype(np.float32)


class SkeletonDS(torch.utils.data.Dataset):
    def __init__(self, X, y, idx, train=False):
        self.X, self.y, self.idx, self.train = X, y, idx, train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = self.X[j].astype(np.float32)
        if self.train:
            x = augment(x)
        x = np.ascontiguousarray(np.transpose(x, (3, 1, 2, 0)))   # C,T,V,M
        return torch.from_numpy(x), int(self.y[j])


def _load_ckpt(p):
    try:
        return torch.load(p, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(p, map_location='cpu')


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
            pred = model(xb).argmax(1)
        correct += (pred == yb).sum().item(); total += yb.numel()
    return 100.0 * correct / total


def run_experiment(tag, labels, split, num_classes, epochs=60, lr=0.1, bs=64,
                   wd=5e-4, warmup=5, workers=2, init_from=None, freeze=False):
    tr_idx, te_idx = SPLITS[split]
    tl = torch.utils.data.DataLoader(SkeletonDS(X, labels, tr_idx, True), bs, shuffle=True,
                                     num_workers=workers, pin_memory=(DEVICE == 'cuda'),
                                     drop_last=True, persistent_workers=workers > 0)
    vl = torch.utils.data.DataLoader(SkeletonDS(X, labels, te_idx, False), bs * 2,
                                     shuffle=False, num_workers=workers,
                                     pin_memory=(DEVICE == 'cuda'))

    model = STGCN(num_classes).to(DEVICE)
    if init_from:
        sd = _load_ckpt(init_from)['model']
        sd = {k: v for k, v in sd.items() if not k.startswith('fc.')}
        print('  init from', os.path.basename(init_from),
              model.load_state_dict(sd, strict=False).missing_keys[:2], '...')
    if freeze:
        for n_, p_ in model.named_parameters():
            if not n_.startswith('fc.'):
                p_.requires_grad = False

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

    last = os.path.join(CKPT_DIR, tag + '_last.pt')
    best_p = os.path.join(CKPT_DIR, tag + '_best.pt')
    start, best, hist = 0, 0.0, []
    if os.path.exists(last):
        ck = _load_ckpt(last)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        start, best, hist = ck['epoch'] + 1, ck['best'], ck['hist']
        print('  resumed at epoch %d (best %.2f%%)' % (start, best))

    def set_lr(ep):
        f = (ep + 1) / warmup if ep < warmup else \
            0.5 * (1 + math.cos(math.pi * (ep - warmup) / max(1, epochs - warmup)))
        for g in opt.param_groups:
            g['lr'] = lr * f
        return lr * f

    print('[%s] split=%s  classes=%d  train=%d  test=%d'
          % (tag, split, num_classes, len(tr_idx), len(te_idx)))
    for ep in range(start, epochs):
        cur_lr = set_lr(ep); model.train(); t0 = time.time(); run = 0.0
        for xb, yb in tl:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()
        acc = evaluate(model, vl)
        hist.append({'epoch': ep, 'loss': run / max(1, len(tl)), 'acc': acc, 'lr': cur_lr})
        torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(),
                    'best': max(best, acc), 'hist': hist}, last)
        if acc > best:
            best = acc; torch.save({'model': model.state_dict(), 'acc': acc}, best_p)
        print('  ep %3d/%d  loss %.3f  acc %6.2f%%  best %6.2f%%  lr %.4f  (%.0fs)'
              % (ep + 1, epochs, hist[-1]['loss'], acc, best, cur_lr, time.time() - t0))

    json.dump(hist, open(os.path.join(RES_DIR, tag + '_history.json'), 'w'), indent=1)
    print('[%s] BEST %.2f%%' % (tag, best))
    return {'tag': tag, 'split': split, 'classes': num_classes,
            'best_acc': best, 'ckpt': best_p}

RESULTS = {}
RES_JSON = os.path.join(RES_DIR, 'phase1_results.json')
if os.path.exists(RES_JSON):
    RESULTS = json.load(open(RES_JSON))
    print('loaded previous results:', list(RESULTS))

def save_results():
    json.dump(RESULTS, open(RES_JSON, 'w'), indent=2)

## Experiment A · Action recognition (utility)

Standard NTU-60 cross-subject benchmark. Sets the accuracy you must not lose when you
add privacy protection in Phase 2. **Expect roughly 80–85%.**

In [ ]:
RESULTS['A_action_xsub'] = run_experiment(
    tag='A_action_xsub', labels=ACT_LABEL, split='xsub',
    num_classes=60, epochs=60, lr=0.1, bs=64)
save_results()

## Experiment B · Re-identification from scratch (leakage)

Identical architecture, identical input — only the label changes from *what* to *who*.
A high number here means skeleton data is **not** anonymous.

In [ ]:
RESULTS['B_reid_rand'] = run_experiment(
    tag='B_reid_rand', labels=SUBJ_LABEL, split='reid_rand',
    num_classes=40, epochs=40, lr=0.1, bs=64)
save_results()

In [ ]:
RESULTS['B_reid_xview'] = run_experiment(
    tag='B_reid_xview', labels=SUBJ_LABEL, split='reid_xview',
    num_classes=40, epochs=40, lr=0.1, bs=64)
save_results()

## Experiment C · Linear probe on the action model (the key result)

The backbone from Experiment A is **frozen**, and a single linear layer is trained on its
256-d features to predict identity. Nothing about identity was ever in its training
objective.

If this scores well above chance (2.5%), then a model built purely for action recognition
has silently encoded who each person is — which is precisely the leak Phase 2 closes.

In [ ]:
RESULTS['C_probe_frozen'] = run_experiment(
    tag='C_probe_frozen', labels=SUBJ_LABEL, split='reid_rand',
    num_classes=40, epochs=30, lr=0.05, bs=128,
    init_from=RESULTS['A_action_xsub']['ckpt'], freeze=True)
save_results()

In [ ]:
# top-5 identity accuracy for the frozen probe (reported alongside top-1)
@torch.no_grad()
def topk(ckpt, labels, split, num_classes, ks=(1, 5), bs=256):
    m = STGCN(num_classes).to(DEVICE)
    m.load_state_dict(_load_ckpt(ckpt)['model']); m.eval()
    ld = torch.utils.data.DataLoader(
        SkeletonDS(X, labels, SPLITS[split][1], False), bs, num_workers=2)
    hit, tot = {k: 0 for k in ks}, 0
    for xb, yb in ld:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
            p = m(xb).topk(max(ks), 1).indices
        for k in ks:
            hit[k] += (p[:, :k] == yb[:, None]).any(1).sum().item()
        tot += yb.numel()
    del m; torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    return {('top%d' % k): 100.0 * hit[k] / tot for k in ks}

for key, split in [('B_reid_rand', 'reid_rand'),
                   ('B_reid_xview', 'reid_xview'),
                   ('C_probe_frozen', 'reid_rand')]:
    if key in RESULTS:
        RESULTS[key].update(topk(RESULTS[key]['ckpt'], SUBJ_LABEL, split, 40))
        print(key, RESULTS[key]['top1'], RESULTS[key]['top5'])
save_results()

## 6 · Table 1 of the paper

In [ ]:
rows = [
    ('Action recognition (utility)', 'A_action_xsub',  'X-Sub',      60),
    ('Re-ID, random split',          'B_reid_rand',    'stratified', 40),
    ('Re-ID, unseen camera',         'B_reid_xview',   'C1,C2->C3',  40),
    ('Re-ID, frozen action features','C_probe_frozen', 'stratified', 40),
]
print('%-32s %-12s %8s %9s %9s' % ('experiment', 'protocol', 'chance', 'top-1', 'top-5'))
print('-' * 76)
lines = []
for name, key, proto, ncls in rows:
    if key not in RESULTS:
        continue
    r = RESULTS[key]
    line = '%-32s %-12s %7.1f%% %8.2f%% %8s' % (
        name, proto, 100.0 / ncls, r['best_acc'],
        ('%.2f%%' % r['top5']) if 'top5' in r else '-')
    print(line); lines.append(line)

open(os.path.join(RES_DIR, 'table1.txt'), 'w').write('\n'.join(lines))
print('\nsaved ->', os.path.join(RES_DIR, 'table1.txt'))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for key, lab in [('A_action_xsub', 'action (utility)'),
                 ('B_reid_rand', 'identity (leakage)'),
                 ('C_probe_frozen', 'identity, frozen probe')]:
    p = os.path.join(RES_DIR, key + '_history.json')
    if os.path.exists(p):
        h = json.load(open(p))
        ax[0].plot([e['epoch'] for e in h], [e['acc'] for e in h], label=lab)
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('accuracy (%)')
ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title('training curves')

ks = [k for _, k, _, _ in rows if k in RESULTS]
ax[1].bar(range(len(ks)), [RESULTS[k]['best_acc'] for k in ks])
ax[1].axhline(2.5, ls='--', c='r', label='identity chance (2.5%)')
ax[1].set_xticks(range(len(ks)))
ax[1].set_xticklabels(['A', 'B-rand', 'B-xview', 'C'][:len(ks)])
ax[1].set_ylabel('accuracy (%)'); ax[1].legend(); ax[1].set_title('Phase 1 summary')
plt.tight_layout()
plt.savefig(os.path.join(RES_DIR, 'phase1_summary.png'), dpi=150)
plt.show()

## 7 · Send me these

1. The printed **Table 1**
2. `results/phase1_results.json`
3. `results/phase1_summary.png`

**How to read the outcome.** If Experiment B lands anywhere near 70–85% and Experiment C
is far above 2.5%, the premise of the paper is confirmed: skeleton data carries identity,
and an action model learns it without being asked. That becomes Phase 2 — the
leakage-suppression training that drives B and C down while holding A steady.

If B comes out unexpectedly low, that is still a result worth having, and we adjust the
framing rather than force the narrative.